# 05. CropHarvest JECAM test collection

Audit carrot, cabbage and potato labels before selecting candidate parcels. Source reliability and date checks remain unchanged.

This notebook keeps the existing reference and imagery paths, campaign IDs, top-up settings, and progress logs. The crop-specific setup applies to new requests; earlier batches retain their saved plans. Organized outputs are linked under `pipeline_outputs/collection/cropharvest/`.


Run All defaults to the separate non-JECAM cabbage supplement. It screens against completed development and JECAM data and writes to its own campaign. Use `collection_cohort = "jecam"` to resume the original JECAM workflow.

## 1. Setup and configuration

Choose a Python 3.12+ kernel, review the settings below, then **Restart Kernel → Run All**. Opening the notebook starts no work. Existing settings, selections, archives and jobs are preserved.

- `reuse`: use local references; satellite collection is enabled separately.
- `download_missing`: obtain missing references within the source limits.
- `offline`: use local references, with no package installation, sign-in or satellite calls. Dependencies must already be installed.

Set `extra_batches` to the number of additional batches to collect before 05–07. Each batch can select up to 50 unused parcels per available crop. Keep the same value to resume or increase it to add batches.

Use `download_missing` on a fresh clone. Enter an Earth Engine-enabled project ID when prompted. All sources target **up to 50 candidate parcels per available crop per round**; existing larger batches remain intact. Saved outputs may describe earlier runs until cells are rerun.


In [13]:
# Review these settings before Run All.
reference_mode = "reuse"  # reuse | download_missing | offline
run_satellite_collection = True  # offline always disables satellite calls
collection_cohort = "cabbage_supplement"  # Or "jecam" for the original workflow.
submit_only = False  # Queue exports. Set False to collect finished results.

from pathlib import Path
import os
import sys
repository_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), Path.cwd())
source_folder = str(repository_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)
from crop_pipeline import notebook_setup, source_workflow
from crop_pipeline.notebook_display import show_table, show_mapping, show_items
reference_options = source_workflow.reference_options(reference_mode)
notebook_setup.prepare_environment(repository_root, install_missing=reference_mode != "offline")
os.chdir(repository_root)

from crop_pipeline.sources import adaptive, annual
from crop_pipeline import progress as pipeline_progress
from crop_pipeline.windows.seasons import file_sha256
settings_path = Path("settings.local.json")
local_settings = notebook_setup.ensure_local_settings(settings_path)
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
annual_archive_root = annual.archive_root(local_settings, report_folder)
run_name = adaptive.COLLECTION_NAME
candidate_parcels_per_crop = source_workflow.ROUND_PARCELS_PER_CROP
extra_batches = 12  # Total extra batch slots for this round.
run_satellite_collection = run_satellite_collection and reference_mode != "offline"
authenticate_now = run_satellite_collection
download_workers = 2
intervals_per_request = 6

from crop_pipeline.sources import cropharvest as cropharvest_source, cropharvest_review
from crop_pipeline.imagery.quality import save_json
from IPython.display import display
source_name = "cropharvest"
inward_buffer_metres = 10  # Polygon interiors, preserving the existing rule.
labels_path = repository_root / "data/external/cropharvest/labels.geojson"
country_boundaries_path = repository_root / "data/external/cropharvest/ne_50m_admin_0_countries.zip"
minimum_reference_year, maximum_reference_year = 2017, 2100
observation_year_overrides = {}  # Explicit source-UID decisions for unresolved dates.

# Resolve account configuration before source transfers; authentication occurs during collection.
earth_engine_project = notebook_setup.project_for_collection(local_settings, settings_path, required=run_satellite_collection)
print(f"Source: {source_name} | References: {reference_mode} | Round limit: {candidate_parcels_per_crop} per crop")
print(f"Inward buffer: {inward_buffer_metres} m | Satellite collection: {run_satellite_collection}")
print("Archive:", annual_archive_root)
print("Backend: Earth Engine batch assets → Parquet transport → annual TIFFs")
notebook_progress = pipeline_progress.start_notebook(
    '03_cropharvest_preparation', source_workflow.STAGES, report_folder.parent / "progress",
    workload={"folder": str(annual_archive_root), "source": source_name},
)


Notebook environment ready: /Users/josedavid/Documents/programming/external-crop-data-classification/.venv/bin/python
Source: cropharvest | References: reuse | Round limit: 50 per crop
Inward buffer: 10 m | Satellite collection: True
Archive: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive
Backend: Earth Engine batch assets → Parquet transport → annual TIFFs


03_cropharvest_preparation: stages:   0%|          | 0/5 [00:00<?, ?stage/s, elapsed 0m 00s; notebook ETA learning (future stages not yet timed)]

## 2. Obtain references

Reuse and validate `labels.geojson`, or obtain it in `download_missing` mode from the [pinned release](https://zenodo.org/records/10251170). New downloads are checksum-checked and saved atomically. No imagery/feature archive is downloaded. Country borders are optional; dataset names provide the existing fallback.


In [14]:
pipeline_progress.begin_stage("Obtain references")
labels_download = cropharvest_source.ensure_labels(
    labels_path, download_if_missing=reference_options["download_if_missing"],
)
labels_revision = "labels.geojson sha256:" + file_sha256(labels_path)
print("Labels:", labels_download["status"], "| features:", labels_download["features"])
reference_note = "Pinned CropHarvest labels; source grades and dates are screening evidence"
pipeline_progress.end_stage()


03_cropharvest_preparation: stages:  20%|██        | 1/5 [00:00<00:02,  1.40stage/s, elapsed 0m 00s; notebook ETA learning (future stages not yet timed)]   

Labels: reused | features: 113893


## 3. Validate references

Normalize crop names, resolve observation years, and apply the existing source-reliability, geometry, duplicate and season checks. Save the full label and year audits; show one crop-level summary. Exclude upstream test labels before preparing footprints.

Dates establish reference years, not planting dates. The helper preserves existing eligibility rules. Polygon interiors use the existing **10 m inward buffer**.


In [15]:
pipeline_progress.begin_stage("Validate references")
# Version the audit independently of notebook layout and collection workload.
run_configuration = {
    "labels_sha256": file_sha256(labels_path),
    "country_boundaries_sha256": file_sha256(country_boundaries_path) if country_boundaries_path.exists() else None,
    "year_range": [minimum_reference_year, maximum_reference_year],
    "observation_year_overrides": observation_year_overrides,
    "review_helper_sha256": file_sha256(Path(cropharvest_review.__file__)),
}
run_folder, run_revision = cropharvest_source.initialize_run(
    annual_archive_root / "cropharvest_label_review", run_name, run_configuration,
)
label_review = cropharvest_review.review_labels(
    labels_path, country_boundaries_path, run_folder,
    minimum_reference_year=minimum_reference_year, maximum_reference_year=maximum_reference_year,
    observation_year_overrides=observation_year_overrides,
)
reviewed_labels = label_review["reviewed_labels"]
show_table(label_review["summary"], title="CropHarvest label screening")
print("Year decisions:", label_review["observation_year_audit"])
print("Screened labels:", label_review["screened_labels_output"])
eligible_labels = reviewed_labels.loc[reviewed_labels["reliable"].eq(True) & ~reviewed_labels["is_test"].eq(True)].copy()
eligible_labels["year"] = eligible_labels["observation_year"].astype(int)
reference_features, footprint_exclusions = [], []
for record in eligible_labels.itertuples():
    try:
        reference_features.append(cropharvest_source.parcel_feature(
            record.uid, record.geometry, record.crop, record.year, record.country,
            record.dataset, labels_revision,
        ))
    except ValueError as error:
        footprint_exclusions.append({"uid": record.uid, "reason": str(error)})
save_json(run_folder / "labels/footprint_exclusions.json", footprint_exclusions)
print("Eligible footprints:", len(reference_features), "| exclusions:", len(footprint_exclusions))
print("Edge handling: 10 m inward buffer on parcel polygons.")
pipeline_progress.end_stage()


03_cropharvest_preparation: Validate references:  20%|██        | 1/5 [00:03<00:14,  3.56s/stage, elapsed 0m 03s; notebook ETA learning (future stages not yet timed)]

Crop,Candidates,Passed,Excluded
Cabbage,190,190,0
Carrot,522,522,0
Potato,670,670,0


03_cropharvest_preparation: stages:  40%|████      | 2/5 [00:04<00:06,  2.11s/stage, elapsed 0m 04s; notebook ETA learning (future stages not yet timed)]             

Year decisions: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive/cropharvest_label_review/runs/adaptive_training/0b4582bef2917256/labels/observation_year_audit.csv
Screened labels: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive/cropharvest_label_review/runs/adaptive_training/0b4582bef2917256/labels/target_crop_labels_screened.csv
Eligible footprints: 1382 | exclusions: 0
Edge handling: 10 m inward buffer on parcel polygons.


## 4. Prepare the collection round

Resume original saved plans and jobs. Fill the round limit only with previously unselected parcels for deficient crop/partition combinations; locked partitions stay unchanged. Prepare masks only for selected candidates and show the combined workload. Source-specific edge handling is preserved.


In [16]:
pipeline_progress.begin_stage("Prepare collection round")
from crop_pipeline.sources import test_cohort
provenance = {"labels_revision": labels_revision, "label_review_revision": run_revision,
              "label_review_folder": str(run_folder), "footprint_exclusions": footprint_exclusions}
if collection_cohort == "cabbage_supplement":
    adaptive_batches = test_cohort.prepare_cabbage_supplement(
        report_folder, annual_archive_root, reference_features, provenance,
        limit=candidate_parcels_per_crop,
    )
elif collection_cohort == "jecam":
    adaptive_batches = test_cohort.prepare(
        report_folder, annual_archive_root, reference_features, provenance,
        limit=candidate_parcels_per_crop, extra_batches=extra_batches,
        candidate_crops=("carrot", "potato"),
    )
else:
    raise ValueError("Choose cabbage_supplement or jecam")
annual_plan, adaptive_context = next(((plan, context) for plan, context in adaptive_batches if plan is not None), (None, adaptive_batches[-1][1]))
print("Active/resumed batches:", len(adaptive_batches))
print("Selected parcels:", sum(context["candidate_parcels"] for plan, context in adaptive_batches if plan is not None))
print("Tiles:", sum(plan["estimate"]["tiles"] for plan, context in adaptive_batches if plan is not None))
show_items(sorted({year for plan, context in adaptive_batches if plan is not None for year in plan["years"]}), title="Years")
pipeline_progress.set_workload({"folder": str(annual_archive_root), "source": source_name,
    "plans": [plan["revision"] for plan, context in adaptive_batches if plan is not None],
    "satellite_collection": run_satellite_collection})
pipeline_progress.end_stage()


03_cropharvest_preparation: stages:  60%|██████    | 3/5 [00:04<00:03,  1.57s/stage, elapsed 0m 04s; notebook ETA learning (future stages not yet timed)]                  

Supplemental cabbage: 43 eligible footprints, 1 overlapping parcel groups excluded
Round 1: resuming saved cropharvest selections and original collection plans.
Round 1: up to 50 candidates per crop in each new batch. Resume/submit 1 batches.
Active/resumed batches: 1
Selected parcels: 43
Tiles: 37
Years: 2019


## 5. Collect annual imagery

Submit or collect existing Earth Engine jobs. Keep the laptop on until Run All returns normally. Submitted jobs continue in the cloud; rerun later to retrieve results and create annual TIFFs.

Cloud masks, daily mosaics, bilinear band alignment and half-month medians are unchanged. Each completed tile retains 24 periods × seven measurements. One progress display follows the active step; detailed task/page messages are saved to its log. Completion describes saved data, not usable-window quality.


In [17]:
pipeline_progress.begin_stage("Collect annual imagery")
collection_result = adaptive.collect_batches(
    adaptive_batches, earth_engine_project,
    enabled=run_satellite_collection, authenticate=authenticate_now,
    submit_only=submit_only,
    download_workers=download_workers, intervals_per_request=intervals_per_request,
)
pipeline_progress.end_stage("skipped" if not run_satellite_collection else "completed" if collection_result["status"] == "completed" else "pending")


03_cropharvest_preparation: stages:  80%|████████  | 4/5 [00:00<00:00, 13640.01stage/s, elapsed 1m 16s; notebook ETA learning (future stages not yet timed)]               

## 6. Summary and next action

Check verified tiles and Earth Engine job states below. Collection notebook 05 can run while 02–04 collect development imagery. Preprocessing notebook 05 checks overlap after completed batches are available.

Annual TIFFs are under `annual_tifs/<source>/<year>/<campaign>/round_<number>/`, beside the configured archive root. Masks and Parquet caches remain separate. See [the workflow guide](../../docs/workflow.md).


In [18]:
pipeline_progress.begin_stage("Summary and next action")
source_workflow.print_summary(
    source_name, adaptive_batches, collection_result, reference_mode,
    inward_buffer_metres, reference_note=reference_note,
)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()


03_cropharvest_preparation: stages: 100%|██████████| 5/5 [00:00<00:00, 808.15stage/s, elapsed 1m 16s; notebook ETA 0m 00s (future stages not yet timed)]                   

cropharvest — completed
References: reuse | Inward buffer: 10 m
Reference coverage: Pinned CropHarvest labels; source grades and dates are screening evidence
Selected parcels: cabbage 43
Batches: cropharvest: completed
Verified local tiles: 37/37 (reported batch ranges)
Cloud jobs: 0 queued, 0 running, 0 ready to download, 0 failed, 0 cancelled, 0 unknown
Campaign: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/adaptive_training/223c7df494ac
Annual TIFF root: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_tifs
Saved registries: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/adaptive_training/223c7df494ac/rounds/0001/cropharvest/source_handoffs/full_year_all_parcels/cropharvest.json
Detail log: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/progress/03_cropharvest_preparation/runs/20261117T023400_4b69d472.log
N

In [19]:
# Set to a folder path to export completed batches for your teammate.
jecam_export_folder = None
if collection_cohort == "jecam" and jecam_export_folder is not None:
    from crop_pipeline.sources.jecam_transfer import export_completed
    for bundle in export_completed(report_folder, jecam_export_folder):
        print("Share JECAM bundle:", bundle)


## Organized outputs

Refresh the linked output folders without moving downloads or changing saved jobs. Run **06_dataset_summary.ipynb** to record all saved batches and selections. Historical outputs above were retained from the original notebook; rerun deliberately to update them.


In [20]:
from crop_pipeline.output_layout import refresh

output_layout = refresh(repository_root)
print("Organized collection outputs:", repository_root / "pipeline_outputs/collection")
print("Next: data_collection/06_dataset_summary.ipynb, then preprocessing/01_imagery_quality.ipynb.")


Organized collection outputs: /Users/josedavid/Documents/programming/external-crop-data-classification/pipeline_outputs/collection
Next: data_collection/06_dataset_summary.ipynb, then preprocessing/01_imagery_quality.ipynb.
